In [ ]:
import sisl
import matplotlib.pyplot as plt
import numpy as np
from TimedependentTransport import TD_Transport, AdaptiveRK4,hbar
from numba import njit
from funcs import diff_central, calc_current
from ase import Atoms
from ase.build import molecule
from ase.visualize import view

In [ ]:
geom_em = sisl.io.siesta.xvSileSiesta('ELEC_TIP.XV').read_geometry()
geom_ep = sisl.io.siesta.xvSileSiesta('ELEC_TIP.XV').read_geometry()
geom_dev = sisl.io.siesta.xvSileSiesta('siesta.XV').read_geometry()

geom_em = geom_em.move(-np.array([0.0, 0.0, geom_em.xyz[:,2].min() - geom_dev.xyz[:,2].min()]))
geom_ep = geom_ep.move(-np.array([0.0, 0.0, geom_ep.xyz[:,2].max() - geom_dev.xyz[:,2].max()]))


In [ ]:
eps  = (1e-6 + 1j  * 1e-2)
line = np.linspace(-0.8, 0.8, 50) + eps
# line = np.array([-8, -4, -2.5, -1.0,  1.0, 2.0 ,3.0, 4.0, 5.0, 6.0, 7.0, 8.0]) + eps
line = np.vstack((line,line))
Test = TD_Transport([geom_em,geom_ep], geom_dev, kT_i = [0.025, 0.025])
Test.Make_Contour(line, 10, pole_mode = 'JieHu2011')
plt.show()
Test.Electrodes(semi_infs = ['-a3', '+a3'], kp = [[1,1,50],[1,1,50]]); 
Test.make_device()
plt.show()
Test.Device.Visualise(axes = [0,2])
plt.show()
view(Test.Device.toASE())


In [ ]:
Test.run_electrodes()
Test.run_device(['TBT.Atoms.Device [ 16 -- 17]'], where = 'TS_TBT')

In [ ]:
Test.read_data(fact = 2.0, NumL=10,
                    fit_mode = 'all', use_analytical_jac = True, 
                    maxiter = 10 ** 6,min_method = 'L-BFGS-B', 
                    ebounds = (line.min()-1 , line.max()+1 ), 
                    wbounds = (0.01, 5.0), 
                    tol = -1.0, options = {'disp':True, 'maxiter': 10**6, 'gtol': 0.1}
              )

In [ ]:
I,J,i,j  = 0,0,5,2 #I,J Blocks from tbtrans, i,j are the subindices in these blocks
Test.Inspect_Lorentzian_fit(1,I,J,i,j, Emin = -2,Emax= 2);

In [ ]:
Test.Inspect_Transmission(0,1)

In [ ]:
Test.get_propagation_quantities()
Test.get_dense_matrices()
sig = Test.sigma
psi = Test.Psi_vec
omega = Test.omegas
no_d = sig.shape[2]

In [ ]:
_ = plt.plot(np.log10(Test.Gl_eig[0,0]).real)

In [ ]:
V = 0.2
@njit
def Bias(t, tp, ts):
    return V * (np.tanh(t / ts) - np.tanh((t - tp) / ts)) / 2
@njit
def fd(x, mu, s):
    return 1 / (1 + np.exp((x - mu) / s))
@njit
def delta(t, a):
    if a == 0:
        return   Bias(t, 20, 1)/2
    elif a == 1:
        return - Bias(t, 20, 1)/2
@njit
def zero_bias(t, a):
    return 0.0
@njit
def zero_dH(t, sigma):
    A = np.zeros((1, no_d, no_d), dtype=np.complex128)
    return A

@njit
def dH(t, sigma):
    # We can put in Hartree correction here like in 
    # "Insights into the Charge Separation Dynamics in Photoexcited
    # Molecular Junctions"
    # 
    # For now we just put a sine somewhere
    A = np.zeros((1, no_d, no_d), dtype=np.complex128)
    A[0,5,5] = np.sin(t) * V/4 * Bias(t, 20, 1)
    return A

f   = Test.make_f_general(parallel = True, fastmath = True)
xi  = Test.xi
Ixi = Test.Ixi

t1, data1 = AdaptiveRK4(
                        f,
                        sig, psi, omega,
                        1e-5, -15.0, 30.0,
                        zero_dH,
                        zero_bias,
                        Test.Ixi,
                        0.01,
                        fixed_mode=False,
                        name="Multithread_RK4",
)

sig          = data1['last sigma']
omega        = data1['last omega']
psi          = data1['last psi']

t2, data2 = AdaptiveRK4(
                        f,
                        sig, psi, omega,
                        1e-5, -5.0, 80.0,
                        dH,
                        delta,
                        Test.Ixi,
                        0.01,
                        fixed_mode=False,
                        name="Multithread_RK4",
)

In [ ]:
dNdt = diff_central(t2, np.array([np.trace(data2['density matrix'][i]) for i in range(len(t2))]))
JL   = data2['current left']
JR   = data2['current right']

In [ ]:
plt.plot(t1, data1['current left'], label = r'$J_L$')
plt.plot(t1, data1['current right'], label = r'$J_R$')
plt.xlabel('time [fs]',size = 18)
plt.ylabel(r'current [$\frac{e}{fs}$]',size = 18)
plt.title('Equilibrium')
plt.legend()

In [ ]:
plt.plot(t2, JL, label = r'$J_L$')
plt.plot(t2, JR, label = r'$J_R$')
plt.xlabel('time [fs]',size = 18)
plt.ylabel(r'current [$\frac{e}{fs}$]',size = 18)
plt.title('With biaspulse, using the equilibrium state as starting point')
plt.legend()

In [ ]:
plt.plot(t2[1:-1], dNdt*0.95, label = r'$0.95*\frac{dN}{dt}$')
plt.plot(t2, JL+ JR, label = r'$J_L + J_R$')
plt.legend()
plt.xlabel('time [fs]',size = 18)
plt.ylabel(r'current [$\frac{e}{fs}$]',size = 18)
plt.title('Check for charge conservation')